# Capteurs en lecture seule

Connexion télémétrique et caméra du plugin. Le poste de guidage charge ce carnet pour la lecture en direct.

In [ ]:
import socket, json, ast, time
HOST='127.0.0.1'; PORT_UDP=2345; PORT_TCP=2347

def management(command,timeout=4):
    if command not in ('get_config','get_static_info'):
        raise ValueError('Cette fonction attend une commande qui retourne des donnees.')
    with socket.create_connection((HOST,PORT_TCP),timeout=timeout) as s:
        s.settimeout(timeout)
        s.sendall(command.encode('ascii'))
        data=b''
        deadline=time.monotonic()+timeout
        while time.monotonic()<deadline and len(data)<1024*1024:
            try: chunk=s.recv(65536)
            except socket.timeout: break
            if not chunk: break
            data+=chunk
            try: return json.loads(data)
            except (ValueError,UnicodeError): pass
            try: return ast.literal_eval(data.decode('utf8'))
            except (ValueError,SyntaxError,UnicodeError): pass
    raise RuntimeError('Reponse management absente/incomplete pour '+command+': '+repr(data[:150]))

def open_telemetry(timeout=6):
    sock=socket.socket(socket.AF_INET,socket.SOCK_DGRAM)
    sock.settimeout(.30)
    sock.connect((HOST,PORT_UDP))
    end=time.monotonic()+timeout
    try:
        while time.monotonic()<end:
            sock.send(b'connect')
            try: data=sock.recv(2**18)
            except socket.timeout: continue
            if data==b'identified': return sock
        raise TimeoutError('Le plugin UDP 2345 ne repond pas : lancer une session de conduite.')
    except Exception:
        sock.close()
        raise

def next_packet(sock,timeout=3,after_steps=None):
    deadline=time.monotonic()+timeout
    while time.monotonic()<deadline:
        try: data=sock.recv(2**18)
        except socket.timeout: continue
        if data==b'disconnect': raise ConnectionError('Connexion telemetrie interrompue')
        if data==b'identified': continue
        try: packet=json.loads(data)
        except (ValueError,UnicodeError): continue
        if not isinstance(packet,dict): continue
        if after_steps is not None and int(packet.get('steps',-1))<=int(after_steps): continue
        packet['_received_unix']=time.time()
        return packet
    raise TimeoutError('Aucun nouvel echantillon telemetrie dans le delai imparti.')

def close_telemetry(sock):
    if sock is None: return
    try: sock.send(b'disconnect')
    except OSError: pass
    sock.close()

In [ ]:
import sys, time, numpy as np, cv2
PLUGIN_PATH=GYM_ROOT/'AssettoCorsaPlugin'/'plugins'/'sensors_par'
if not PLUGIN_PATH.is_dir():
    raise FileNotFoundError('Plugin Python source manquant : '+str(PLUGIN_PATH))
if str(PLUGIN_PATH) not in sys.path:
    sys.path.insert(0,str(PLUGIN_PATH))
from dual_buffer import GrabberSharedMemoryDualBuffer

class CameraReader:
    def __init__(self,config,age_limit=.55):
        if not config.get('screen_capture_enable'):
            raise RuntimeError('screen_capture_enable=False dans le plugin installe.')
        self.cam=GrabberSharedMemoryDualBuffer(int(config['final_image_width']),
                      int(config['final_image_height']),str(config['color_mode']))
        self.age_limit=float(age_limit)
        self.last_seq=-1
    def read(self,timeout=2,require_new=True):
        deadline=time.monotonic()+float(timeout)
        last_err='Aucune image stable et fraiche'
        while time.monotonic()<deadline:
            try:
                seq,stamp,arr=self.cam.get_timed_image()
                age=time.time()-float(stamp)
                if not np.isfinite(age) or age<-.15 or age>self.age_limit:
                    last_err='Image perimee ou horloge incoherente (age %.3f s)'%age
                elif require_new and int(seq)<=self.last_seq:
                    last_err='Image non actualisee (seq %s)'%seq
                else:
                    self.last_seq=int(seq)
                    return int(seq),float(stamp),arr
            except (RuntimeError,ValueError) as e:
                last_err=str(e)
            time.sleep(.015)
        raise TimeoutError('Capture indisponible : '+last_err)

In [ ]:
class OverlayTrajectoire:
    """Fenêtre Windows transparente alignée sur la zone cliente capturée par MSS."""
    def __init__(self):
        import ctypes
        import sys
        import tkinter as tk
        import win32con
        import win32gui
        if sys.getwindowsversion().build<19041:
            raise RuntimeError("Utiliser AFFICHAGE='notebook' sur cette version de Windows")
        self.fenetre = tk.Tk()
        self.gui = win32gui
        try:
            self.fenetre.withdraw()
            self.fenetre.overrideredirect(True)
            self.fenetre.attributes('-topmost',True)
            self.fenetre.configure(bg='#010101')
            self.fenetre.attributes('-transparentcolor','#010101')
            self.canvas = tk.Canvas(self.fenetre,bg='#010101',highlightthickness=0)
            self.canvas.pack(fill='both',expand=True)
            self.fenetre.geometry('1x1+0+0')
            self.fenetre.update_idletasks()
            self.hwnd = win32gui.GetAncestor(self.fenetre.winfo_id(),2)
            style = win32gui.GetWindowLong(self.hwnd,win32con.GWL_EXSTYLE)
            win32gui.SetWindowLong(self.hwnd,win32con.GWL_EXSTYLE,
                style|win32con.WS_EX_TRANSPARENT|win32con.WS_EX_LAYERED|0x08000000)
            from ctypes import wintypes
            exclure = ctypes.windll.user32.SetWindowDisplayAffinity
            exclure.argtypes = (wintypes.HWND,wintypes.DWORD)
            if not exclure(self.hwnd,0x11):
                raise RuntimeError("Windows ne peut exclure la ligne de la capture. Choisir AFFICHAGE='notebook'.")
            self.zone = None
        except BaseException:
            self.fermer()
            raise

    def tracer(self, plan):
        import win32con
        jeu = self.gui.FindWindow(None,'Assetto Corsa')
        if not jeu:
            self.effacer()
            return
        x,y = self.gui.ClientToScreen(jeu,(0,0))
        _,_,w,h = self.gui.GetClientRect(jeu)
        if min(w,h)<=0:
            self.effacer()
            return
        if self.zone!=(x,y,w,h):
            self.fenetre.deiconify()
            self.gui.SetWindowPos(self.hwnd,win32con.HWND_TOPMOST,x,y,w,h,
                                  win32con.SWP_NOACTIVATE|win32con.SWP_SHOWWINDOW)
            self.zone = x,y,w,h
        self.canvas.delete('all')
        points = (plan.xy*np.array([w-1,h-1])).ravel().tolist()
        self.canvas.create_line(*points,fill='#00f29b',width=4)
        self.canvas.create_text(points[0],points[1]+18,text=f'{plan.cible_kmh:.0f} km/h',
                                fill='white',font=('Arial',14,'bold'))
        self.fenetre.update()

    def effacer(self):
        self.fenetre.withdraw()
        self.fenetre.update()
        self.zone = None

    def fermer(self):
        self.fenetre.destroy()

In [ ]:
from IPython.display import display, Image
import time

def verifier_fenetre_jeu():
    import win32gui
    jeu = win32gui.FindWindow(None,'Assetto Corsa')
    if not jeu or win32gui.GetForegroundWindow()!=jeu or win32gui.IsIconic(jeu):
        raise GuidageIndisponible("Remettre Assetto Corsa au premier plan")

def afficher_hud(perception, planificateur, duree):
    config = management('get_config')
    if config.get('vjoy_executed_by_server') is not False:
        raise RuntimeError("Le mode HUD exige vjoy_executed_by_server=False")
    vue = display("Basculer sur Assetto Corsa. Afficher le notebook sur un second écran.",display_id=True)
    if AFFICHAGE not in ('notebook','superposition'):
        raise ValueError("AFFICHAGE inconnu")
    overlay = None
    camera = CameraReader(config)
    sock = None
    compte = dict(images=0,plans=0,indisponibles=0)
    try:
        if AFFICHAGE=='superposition':
            overlay = OverlayTrajectoire()
        sock = open_telemetry()
        fin = time.monotonic()+duree
        while time.monotonic()<fin:
            try:
                verifier_fenetre_jeu()
                # Vider les paquets accumulés pendant le rendu du notebook.
                sock.setblocking(False)
                try:
                    for _ in range(4096):
                        sock.recv(2**18)
                except BlockingIOError:
                    pass
                finally:
                    sock.settimeout(.30)
                raw = next_packet(sock,timeout=.8)
                _,stamp,image = camera.read(timeout=.8)
                ts = float(raw['timestamp_unix'])
                if not np.isfinite(ts) or abs(stamp-ts)>.4 or abs(time.time()-ts)>.55:
                    raise GuidageIndisponible("Paire image/télémétrie périmée")
                verifier_fenetre_jeu()
                compte['images'] += 1
                _,p = perception.observer(image)
                plan = planificateur.proposer(p,raw['speed'],raw['angular_velocity_y'])
                if time.time()-min(stamp,ts)>.55:
                    raise GuidageIndisponible("Guidage périmé après inférence")
                if overlay is None:
                    ok,jpg = cv2.imencode('.jpg',dessiner(image,plan))
                    if not ok:
                        raise RuntimeError("Encodage de l'afficheur impossible")
                    vue.update(Image(data=jpg.tobytes()))
                else:
                    overlay.tracer(plan)
                compte['plans'] += 1
            except (GuidageIndisponible,TimeoutError) as exc:
                if overlay is not None:
                    overlay.effacer()
                vue.update(f"Guidage indisponible — {exc}")
                compte['indisponibles'] += 1
                time.sleep(.1)
    finally:
        if overlay is not None:
            overlay.fermer()
        vue.update("Afficheur arrêté.")
        try:
            close_telemetry(sock)
        finally:
            for nom in ('dual_shm','meta_shm'):
                mapping = getattr(camera.cam,nom,None)
                if mapping is not None:
                    mapping.close()
    return compte